<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 数据加载流水线总结

完整的章节代码位于 [ch02.ipynb](./ch02.ipynb)。

本 notebook 包含核心要点——去除了中间步骤的数据加载流水线。

本 notebook 中使用的包：

**数据加载流水线总览（5 个步骤）：**

```
原始文本 → ① 分词 → ② 滑动窗口 → ③ Batch 化 → ④ Token Embedding + Positional Embedding
                                                                          ↓
                                                               input_embeddings (8, 4, 256)
                                                                          ↓
                                                                    送入 Transformer
```

| 步骤 | 做什么 | 关键参数 |
|---|---|---|
| ① 分词 | 文本 → token ID 列表 | tiktoken GPT-2 BPE |
| ② 滑动窗口 | 切成 (input, target) 对 | max_length, stride |
| ③ Batch 化 | 多个样本捆成 batch | batch_size, shuffle |
| ④ Token Embedding | token ID → 语义向量 | vocab_size, output_dim |
| ⑤ Positional Embedding | 加上位置信息 | context_length, output_dim |

In [ ]:
# NBVAL_SKIP
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.4.0
tiktoken version: 0.7.0


In [2]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


# ═══════════════════════════════════════════════════════════════════
# 第 1+2 步：GPTDatasetV1 — 将文本分词并用滑动窗口切分为 (input, target) 对
# ═══════════════════════════════════════════════════════════════════
# 参数:
#   txt        — 原始文本字符串
#   tokenizer  — BPE 分词器（tiktoken GPT-2）
#   max_length — 每个样本的 token 数（上下文窗口大小）
#   stride     — 窗口滑动步长；stride = max_length 时无重叠
#
# 数据流: txt → encode → [token1, token2, ...] → 滑动窗口切片 → (input_chunk, target_chunk)
#   input_chunk  = token_ids[i     : i+max_length]    ← 输入序列
#   target_chunk = token_ids[i+1   : i+max_length+1]  ← 目标序列（右移 1 位）
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []    # 存放所有输入 chunk
        self.target_ids = []   # 存放所有目标 chunk（与 input_ids 一一对应）

        # 第 1 步：将整篇文本一次性编码为 token ID 列表
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # 第 2 步：滑动窗口切分 — 每次前进 stride 步，切出 max_length 长度的子序列
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]            # 输入：从 i 开始取 max_length 个 token
            target_chunk = token_ids[i + 1 : i + max_length + 1]  # 目标：从 i+1 开始取（右移 1 位）
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


# ═══════════════════════════════════════════════════════════════════
# 第 3 步：create_dataloader_v1 — 封装「文本 → Dataset → DataLoader」流程
# ═══════════════════════════════════════════════════════════════════
# 返回可直接迭代的 DataLoader，实现自动 batch 化、shuffle、多进程加载
def create_dataloader_v1(txt, batch_size, max_length, stride,
                         shuffle=True, drop_last=True, num_workers=0):
    # 初始化 BPE 分词器
    tokenizer = tiktoken.get_encoding("gpt2")

    # 用 GPTDatasetV1 将文本切分为滑动窗口样本
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # 用 DataLoader 包装，实现自动 batch 化
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


# ═══════════════════════════════════════════════════════════════════
# 第 4+5 步：Embedding 层配置 + 数据加载
# ═══════════════════════════════════════════════════════════════════

# 读取原始文本
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Embedding 参数配置
vocab_size = 50257       # GPT-2 词汇表大小
output_dim = 256        # 每个 token 的嵌入维度
context_length = 1024   # 最大上下文长度

# Token Embedding: token ID → 256 维语义向量
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
# Positional Embedding: 位置索引 → 256 维位置向量
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# 创建 DataLoader: stride=max_length=4（无重叠），batch_size=8
batch_size = 8
max_length = 4
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=batch_size,
    max_length=max_length,
    stride=max_length
)

In [3]:
# 遍历 DataLoader，取出第一个 batch 做演示
# 完整训练时去掉 break，遍历所有 batch
for batch in dataloader:
    x, y = batch    # x: inputs (8, 4), y: targets (8, 4)

    # 第 4 步: Token Embedding — 每个 token ID → 256 维语义向量
    # x (8, 4) → token_embeddings (8, 4, 256)
    token_embeddings = token_embedding_layer(x)

    # 第 5 步: Positional Embedding — 位置 [0,1,2,3] → 256 维位置向量
    # torch.arange(max_length) = [0, 1, 2, 3]
    # pos_embeddings shape (4, 256)
    pos_embeddings = pos_embedding_layer(torch.arange(max_length))

    # 最终输入嵌入 = 语义 + 位置（逐元素相加，广播）
    # (8, 4, 256) + (4, 256) → (8, 4, 256)
    input_embeddings = token_embeddings + pos_embeddings

    break  # 只取第一个 batch 做演示

In [4]:
# 验证最终输出 shape: (batch_size, max_length, output_dim) = (8, 4, 256)
# 这就是送入 Transformer 的输入张量
print(input_embeddings.shape)

torch.Size([8, 4, 256])
